In [1]:
import cv2 as cv
import torch
from ultralytics import YOLO
from collections import deque
import time

In [ ]:
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

In [2]:
model=YOLO("yolo11n.pt")
cap=cv.VideoCapture("../data/3min.mp4")


In [5]:
if not cap.isOpened():
    print("Не удалось открыть видео — проверьте путь или кодек")

In [3]:
colors = {
    "car": (0, 255, 0),
    "bus": (255, 0, 0),
}

In [4]:
#quick remove last item (pop(0))
fps_history=deque(maxlen=30)

In [ ]:
_,frame=cap.read()
print(frame.shape)
cv.rectangle(frame,(250,600),(720,850),(255,255,0),3)
cv.imshow("1",frame)
cv.waitKey(0)

(1280, 720, 3)


In [3]:
x1_roi,y1_roi,x2_roi,y2_roi=250,600,720,850

In [10]:
cv.imshow("1",frame[y1_roi:y2_roi,x1_roi:x2_roi])
cv.waitKey(0)

-1

In [ ]:
cap=cv.VideoCapture("../data/3min.mp4")
prev_time=time.time()
while cap.isOpened():
    ret, frame= cap.read()
    results = model.predict(frame[y1_roi:y2_roi,x1_roi:x2_roi],classes=[2,3,5,7], verbose=False)[0]
    # results = model.track(frame[y1_roi:y2_roi,x1_roi:x2_roi],persist=True, classes=[2,3,5,7], verbose=False)[0]
    cur_time=time.time()
    fps_history.append(1/(cur_time-prev_time))
    prev_time=time.time()
    avg_fps=sum(fps_history)/len(fps_history)
    cv.putText(frame, f"FPS: {avg_fps:.1f}",(10,30), cv.FONT_HERSHEY_SIMPLEX,1,(0,255,0),2)

    for result in results.boxes:
        x1,y1, x2,y2=map(int,result.xyxy[0])
        #transform coord from roi_frame to origin frame
        x1, x2 = x1 + x1_roi, x2 + x1_roi
        y1, y2 = y1 + y1_roi, y2 + y1_roi
        
        cv.rectangle(frame,(x1,y1),(x2,y2),(255,0,0),3)
        class_id=int(result.cls[0])
        class_name=model.names[class_id]
        conf=result.conf[0]
        color_ob=colors.get(class_name,(255,255,255))
        track_id = int(result.id[0]) if result.id is not None else -1
        ob_label=f"{class_name} #{track_id}  {conf:.2f}"
        

        cv.putText(frame, ob_label,(x1,y1-8), cv.FONT_HERSHEY_SIMPLEX, 0.5, color_ob, 2)
        cv.imshow("frame",frame)
    if cv.waitKey(1)==ord("q"):
        print("avg_fps")
        break
cap.release()
cv.destroyAllWindows()


requirements: Ultralytics requirement ['lap>=0.5.12'] not found, attempting AutoUpdate...
Using Python 3.13.11 environment at: d:\project\traffic_control\.venv
Resolved 2 packages in 692ms
 Downloaded lap
Prepared 1 package in 191ms
         If the cache and target directories are on different filesystems, hardlinking may not be supported.
         If this is intentional, set `export UV_LINK_MODE=copy` or use `--link-mode=copy` to suppress this warning.
Installed 1 package in 8ms
 + lap==0.5.13

requirements: AutoUpdate success  1.2s
WARNING requirements: Restart runtime or rerun command for updates to take effect

avg_fps


In [37]:
print(results.speed)

{'preprocess': 1.1560000020836014, 'inference': 34.292999996978324, 'postprocess': 0.8593999991717283}


In [30]:
import torch
print(torch.get_num_threads())

8


In [ ]:
results = model.predict(source="../data/3min.mp4", save=True, stream=True)

In [ ]:
for result in results:
    xywh = result.boxes.xywh  # center-x, center-y, width, height
    xywhn = result.boxes.xywhn  # normalized
    xyxy = result.boxes.xyxy  # top-left-x, top-left-y, bottom-right-x, bottom-right-y
    xyxyn = result.boxes.xyxyn  # normalized
    names = [result.names[cls.item()] for cls in result.boxes.cls.int()]  # class name of each box
    confs = result.boxes.conf  # confidence score of each box

In [ ]:
model.names